In [1]:
from ollama import Client
import os
from dotenv import load_dotenv
import json


In [2]:
load_dotenv()
ollama_api_key = os.getenv('OLLAMA_API_KEY')

In [3]:
headers = {
    'Authorization': f'Bearer {ollama_api_key}'
}

In [4]:
fake_events ={
        'hyderabad':'No new events for today',
        'bangalore':'Election campaign is going on',
        'chennai':'2 Ranji matches are played today',
        'goa':'fashion events planned '
    }

def get_latest_events_based_on_city(city:str)->str:
    ''' get events based on city'''

    if city.lower() in fake_events.keys():
        return fake_events[city.lower()]
    else:
        return f'no information available for this city : {city}'
    

    

In [5]:
tools = [
    {
        'type':'function',
        'function':{
            'name':'get_latest_events_based_on_city',
            'description': 'Get latest events based on city provided by user',
            'parameters':{
                'type':'object',
                'properties':{
                    'city':{
                        'type':'strig',
                        'description':'name of the city, eg: Hyderabad',
                    }
                },
                'required':['city']
            }
        }
    }
]

available_tools = {
    "get_latest_events_based_on_city": get_latest_events_based_on_city,
}

In [6]:
def get_messages(prompt):
    messages = [
        {
            'role':'assistant',
            'content':'Answer the user question in 2 lines'
        },
        {
            'role':'user',
            'content':prompt
        }
    ]
    return messages


In [ ]:
client = Client(host = 'https://ollama.com',
                headers = headers)

def ollama_chat(prompt, messages):
    response = client.chat(
        model = 'gemma4:31b-cloud',
        messages=messages, 
        tools = tools
    )
    
    print(f'response : {response['message']['content']}')
    try:
        if response['message']['tool_calls'] is not None:
            tool_call = response['message']['tool_calls'][0]
            function_name = tool_call.function.name
            
            arguments = tool_call.function.arguments
            print(f'arguments - {arguments}')
            print(f'calling {function_name} with {arguments}')
            result = available_tools[function_name](**arguments)
            print(result)

            ### passing the result back to LLM
            messages.append(response['message'])
            messages.append({
                'role':'tool',
                # 'tool_call_id':tool_call.id,
                'content':result
            })

            final_reposne = client.chat(
                model = 'gemma4:31b-cloud',
                messages=messages, 
                tools = tools
            )

            print(f'final_reposne : {final_reposne['message']['content']}')
    except Exception as e:
        print(e)


In [10]:
prompt = 'start asking'
count = 0
while prompt.lower() != 'quit' and count < 3:
    prompt = input('Ask Question or enter quit to stop session ')
    if prompt.lower() != 'quit':
        messages = get_messages(prompt)
        ollama_chat(prompt ,messages)
        count = count + 1
    else:
        print(f'quiting session')
    

AttributeError: 'function' object has no attribute 'completions'